# CIS*4020 Group 10 - Project

**Members**
- Aeyan Siddiqi
- Ankush Madharha
- Saad Shafiq
- Aryan Qureshi 

**Data:** download "Predict Students' Dropout and Academic Success" data and point `DATA_DIR` at the CSVs
(`https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success`).

# Gather Data

In [188]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 40)
print("pandas", pd.__version__)

DATA_DIR = "content/"   # <-- put the downloaded .csv files here

pandas 3.1.1


In [189]:
FILE = "data.csv"

def load_data():
    missing = not os.path.exists(DATA_DIR)
    if missing:
        raise FileNotFoundError(
            "Couldn't find file(s)."
            "Set DATA_DIR to the folder containing the CSVs."
        )
    print("[info] loading real data from", DATA_DIR)
    return pd.read_csv(os.path.join(DATA_DIR, FILE), sep=";")

raw_data = load_data()
print(f"{FILE}: {raw_data.shape}")
raw_data.head()

[info] loading real data from content/
data.csv: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,5,9,127.3,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,3,3,142.5,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,9,9,124.8,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,5,3,119.6,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,9,9,141.5,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## Clean Data

Each step ahead will state a decision and justification for that decision.

### 1. Fix column names
- Fix `Nacionality` typo and replace with `Nationality`, easier to read.
- Strip the \t in `Daytime/evening attendance\t`, probably a leftover tab character.
- convert all columns to snake case ('snake_case') for more standard variable names.

Overall these changes make the data much easier to work with in code and while analyzing.

In [190]:
import re

def fix_name(name: str) -> str:
    # strip gets rid of \t and then do a replace for the type
    name = name.strip().replace("Nacionality", "Nationality").replace("'s", "")
    #snake case
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()

df = raw_data.copy()
# fix them
df.columns = [fix_name(c) for c in df.columns]

assert df.columns.is_unique
list(df.columns)


['marital_status',
 'application_mode',
 'application_order',
 'course',
 'daytime_evening_attendance',
 'previous_qualification',
 'previous_qualification_grade',
 'nationality',
 'mother_qualification',
 'father_qualification',
 'mother_occupation',
 'father_occupation',
 'admission_grade',
 'displaced',
 'educational_special_needs',
 'debtor',
 'tuition_fees_up_to_date',
 'gender',
 'scholarship_holder',
 'age_at_enrollment',
 'international',
 'curricular_units_1st_sem_credited',
 'curricular_units_1st_sem_enrolled',
 'curricular_units_1st_sem_evaluations',
 'curricular_units_1st_sem_approved',
 'curricular_units_1st_sem_grade',
 'curricular_units_1st_sem_without_evaluations',
 'curricular_units_2nd_sem_credited',
 'curricular_units_2nd_sem_enrolled',
 'curricular_units_2nd_sem_evaluations',
 'curricular_units_2nd_sem_approved',
 'curricular_units_2nd_sem_grade',
 'curricular_units_2nd_sem_without_evaluations',
 'unemployment_rate',
 'inflation_rate',
 'gdp',
 'target']

### 2. Drop Enrolled Students
We are removing the `Enrolled` students as they do not pretain to what our models are trying to predict. Our question is `Dropout` or `Graduation`, enrolled students have no effect on the final outcome of this project.

In [191]:
n_before = len(df)

# the target is the variable that decides
df = df[df["target"] != "Enrolled"].copy()

# we will create a new column for the binary dropout/not-dropout
df["dropout"] = (df["target"] == "Dropout").astype(int)
df = df.drop(columns="target").reset_index(drop=True)

# print out results
print(f"Dropped rows: {n_before - len(df)}")
print(df["dropout"].value_counts().rename({0: "Graduate", 1: "Dropout"}))
print(f"Dropout rate: {df['dropout'].mean():.2%}")

Dropped rows: 794
dropout
Graduate    2209
Dropout     1421
Name: count, dtype: int64
Dropout rate: 39.15%


### 3. Remove Leakage
There are 12 `curricular_units_x` columns that declare the performance for semesters per student. The issue is that these will quickly fall to 0 if a student drops out, as they do not attend the class. This will leak the result of the student to the models, and increase accuracy without giving us any meaningful insights.

In [192]:
# get and drop them
non_curricular_cols = [c for c in df.columns if c[:16] != 'curricular_units']
df = df[non_curricular_cols]

list(df.columns)

['marital_status',
 'application_mode',
 'application_order',
 'course',
 'daytime_evening_attendance',
 'previous_qualification',
 'previous_qualification_grade',
 'nationality',
 'mother_qualification',
 'father_qualification',
 'mother_occupation',
 'father_occupation',
 'admission_grade',
 'displaced',
 'educational_special_needs',
 'debtor',
 'tuition_fees_up_to_date',
 'gender',
 'scholarship_holder',
 'age_at_enrollment',
 'international',
 'unemployment_rate',
 'inflation_rate',
 'gdp',
 'dropout']

### 4. Map Categories
- **Occupations:** There are many detailed occupations groups, for better predicitions, we will group them into larger containing groups. 
- **Qualifications:** 17–34 codes describe one education ladder. We will group them into `none / basic / secondary / post-secondary / higher` + `Unknown`.
- **Nationality:** Due to this dataset we know that it is 97.5% Portuguese, the other 20 nationalities have ≤ 38 rows each, and the column duplicates `international` (international only defines if `nationality != 1`). We will drop it.

In [193]:
OCCUPATION_GROUPS = {
    0: "Student", 1: "Managers/Directors", 2: "Professionals/Scientists", 3: "Technicians",
    4: "Administrative", 5: "Services/Sales", 6: "Agriculture", 7: "Skilled trades",
    8: "Machine operators", 9: "Unskilled", 10: "Armed forces",
}

QUALIFICATION_LEVELS = {
    "none":           [35, 36],
    "basic":          [9, 10, 11, 12, 14, 15, 19, 26, 27, 29, 30, 37, 38],   # < completed secondary
    "secondary":      [1, 13, 18, 20, 22, 25, 31, 33],                       # 12th year / equivalent
    "post-secondary": [39, 42],                                              # technological / professional
    "higher":         [2, 3, 4, 5, 6, 40, 41, 43, 44],                       # bachelor / doctorate
    "Unknown":        [34],
}

def get_occupation_group(code: int) -> str:
    if code in (90, 99):
        return "Unknown"
    if code > 100: # map to major group (101-103 -> armed forces)
        code = (code - 100) // 10 or 10
    return OCCUPATION_GROUPS[code]

QUAL_MAP = {code: lvl for lvl, codes in QUALIFICATION_LEVELS.items() for code in codes}

# map occupations and qualifications

for col in ("mother_occupation", "father_occupation"):
    df[col] = df[col].map(get_occupation_group)
for col in ("mother_qualification", "father_qualification", "previous_qualification"):
    unmapped = set(df[col].unique()) - set(QUAL_MAP)
    assert not unmapped, f"{col}: unmapped codes {unmapped}"
    df[col] = df[col].map(QUAL_MAP)
    
# drop nationality (I guess we didn't need to fix the typo)
df = df.drop(columns="nationality")

# count each and compare
for col in ("mother_occupation", "father_occupation", "mother_qualification", "father_qualification", "previous_qualification"):
    print(col, df[col].value_counts().to_dict())


mother_occupation {'Unskilled': 1338, 'Administrative': 679, 'Services/Sales': 441, 'Technicians': 275, 'Professionals/Scientists': 246, 'Skilled trades': 228, 'Student': 143, 'Managers/Directors': 87, 'Unknown': 85, 'Agriculture': 77, 'Machine operators': 29, 'Armed forces': 2}
father_occupation {'Unskilled': 861, 'Skilled trades': 562, 'Services/Sales': 414, 'Technicians': 324, 'Administrative': 319, 'Machine operators': 267, 'Armed forces': 217, 'Agriculture': 196, 'Professionals/Scientists': 150, 'Student': 125, 'Managers/Directors': 113, 'Unknown': 82}
mother_qualification {'basic': 2151, 'secondary': 867, 'higher': 469, 'Unknown': 127, 'post-secondary': 10, 'none': 6}
father_qualification {'basic': 2434, 'secondary': 742, 'higher': 321, 'Unknown': 109, 'post-secondary': 14, 'none': 10}
previous_qualification {'secondary': 3019, 'basic': 212, 'higher': 207, 'post-secondary': 192}


### 5. Final Type Changes
We are going to explicitly translate some column types to categories so they cannot be mistaked for ratio or interval values by the model later.

In [ ]:
QUAL_LEVELS = ["none", "basic", "secondary", "post-secondary", "higher", "Unknown"]
binary_cols  = ["daytime_evening_attendance", "displaced", "educational_special_needs", "debtor",
                "tuition_fees_up_to_date", "gender", "scholarship_holder", "international", "dropout"]
nominal_cols = ["course", "marital_status", "application_mode", "mother_occupation", "father_occupation"]
qual_cols    = ["mother_qualification", "father_qualification", "previous_qualification"]

df[binary_cols] = df[binary_cols].astype("int8")
for col in nominal_cols:
    df[col] = df[col].astype("category")
for col in qual_cols:
    df[col] = pd.Categorical(df[col], categories=QUAL_LEVELS) 

# print final result of cleaning
df.head()

,marital_status,application_mode,application_order,course,daytime_evening_attendance,previous_qualification,previous_qualification_grade,mother_qualification,father_qualification,mother_occupation,father_occupation,admission_grade,displaced,educational_special_needs,debtor,tuition_fees_up_to_date,gender,scholarship_holder,age_at_enrollment,international,unemployment_rate,inflation_rate,gdp,dropout
0,1,17,5,171,1,secondary,122.0,basic,basic,Services/Sales,Unskilled,127.3,1,0,0,1,1,0,20,0,10.8,1.4,1.74,1
1,1,15,1,9254,1,secondary,160.0,secondary,higher,Technicians,Technicians,142.5,1,0,0,0,1,0,19,0,13.9,-0.3,0.79,0
2,1,1,5,9070,1,secondary,122.0,basic,basic,Unskilled,Unskilled,124.8,1,0,0,0,1,0,19,0,10.8,1.4,1.74,1
3,1,17,2,9773,1,secondary,122.0,basic,basic,Services/Sales,Technicians,119.6,1,0,0,1,0,0,20,0,9.4,-0.8,-3.12,0
4,2,39,1,8014,0,secondary,100.0,basic,basic,Unskilled,Unskilled,141.5,0,0,0,1,0,0,45,0,13.9,-0.3,0.79,0
